#  Data Pipeline

**Section 1 of the assignment: data handling and memory management.**

This notebook (a) confirms the raw dataset covers the week we need to forecast
(Dec 16-22, 2013), (b) demonstrates the memory cost of a naive full load versus
`forecasting.DataLoader`'s two-pass chunked aggregation, (c) runs the loader over
all 62 raw daily files to build the processed dataset every later notebook reads,
and (d) ranks Milan's 10,000 grid squares by total traffic to identify the three
squares used throughout the rest of the project.

All heavy logic lives in the `forecasting` package (`forecasting/data.py`); this
notebook only calls it and narrates what came back.

In [6]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "forecasting").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import json
import time

import pandas as pd

from forecasting.data import DataLoader, naive_load_day, measure_peak_memory

RAW_DIR = ROOT / "data" / "raw"
DAILY_DIR = ROOT / "data" / "processed" / "daily"
COMBINED_PATH = ROOT / "data" / "processed" / "internet_traffic.parquet"
RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

In [7]:
raw_files = sorted(RAW_DIR.glob("sms-call-internet-mi-*.txt"))
dates = [f.stem.replace("sms-call-internet-mi-", "") for f in raw_files]
print(f"{len(raw_files)} raw files found, {dates[0]} .. {dates[-1]}")

required_week = {f"2013-12-{d:02d}" for d in range(16, 23)}
missing = required_week - set(dates)
if missing:
    raise RuntimeError(f"Missing raw files for the Dec 16-22 evaluation week: {sorted(missing)}")
print("Dec 16-22 evaluation week: all 7 daily files present.")

62 raw files found, 2013-11-01 .. 2014-01-01
Dec 16-22 evaluation week: all 7 daily files present.


**Confirmed.** All 62 daily files are present, spanning 2013-11-01 through
2014-01-01, which fully covers the December 16-22 week required for the
forecasting evaluation in `02_experiments.ipynb`. Nothing needs to be
downloaded or patched before continuing.

In [8]:
# Memory comparison: naive single read_csv vs. DataLoader's two-pass approach,
# each measured in an isolated subprocess (fair, clean-slate comparison).
sample_file = raw_files[0]
tmp_out = RESULTS_DIR / "_memory_demo.parquet"

naive_peak = measure_peak_memory(naive_load_day, sample_file)
optimized_peak = measure_peak_memory(DataLoader().process_day, sample_file, tmp_out)
tmp_out.unlink(missing_ok=True)

memory_table = pd.DataFrame([
    {"approach": "Naive (read_csv, all columns, default dtypes)", "peak_MB": naive_peak / 1e6},
    {"approach": "DataLoader (two-pass, chunked, downcast, preallocated array)", "peak_MB": optimized_peak / 1e6},
])
memory_table

,approach,peak_MB
0,"Naive (read_csv, all columns, default dtypes)",453.783552
1,"DataLoader (two-pass, chunked, downcast, preal...",206.524416


**Interpretation.** TThe naive approach would read all data in the file into memory, which would make the process consume linear memory with respect to the file size, which is completely unmanageable on a laptop for all 62 files. The 'process_day' method of 'DataLoader' will chunk the input and accumulate values directly into a small, pre-allocated float32 array, allowing memory use to never increase with file size.

In [9]:
# Build the full processed dataset: one two-pass aggregation per raw day,
# skipping any day already processed (idempotent), then a single combine.
loader = DataLoader()

t0 = time.time()
stats = loader.build_all(RAW_DIR, DAILY_DIR)
build_seconds = time.time() - t0
print(f"Processed {len(stats)} new day(s) in {build_seconds:.1f}s "
      f"({len(list(DAILY_DIR.glob('*.parquet')))} day-files total on disk).")

t0 = time.time()
combined = loader.combine(DAILY_DIR, COMBINED_PATH)
print(f"Combined into {COMBINED_PATH.name}: {combined.shape} in {time.time()-t0:.1f}s")
combined.head()

Processed 0 new day(s) in 0.0s (62 day-files total on disk).
Combined into internet_traffic.parquet: (89280000, 3) in 71.4s


,square_id,internet_traffic,timestamp
0,1,11.028366,2013-11-01 00:00:00
1,1,11.127101,2013-11-01 00:10:00
2,1,10.892771,2013-11-01 00:20:00
3,1,8.622424,2013-11-01 00:30:00
4,1,8.009928,2013-11-01 00:40:00


**Interpretation.** Each day's worth of data is processed individually, only the missing outputs being generated, after which `combine()` merges the 62 Parquet files into one sorted by `(square_id, timestamp)`. The compact `int16`/`float32` format makes for a dataset that fits in memory and allows full memory analysis and fast per-square filtering.

In [10]:
totals = combined.groupby("square_id")["internet_traffic"].sum().sort_values(ascending=False)
top3 = totals.head(3)
print("Top 3 squares by total internet traffic (Nov 1 - Jan 1):")
print(top3)

top_squares_info = {
    "top3_square_ids": [int(s) for s in top3.index],
    "top3_totals": {int(s): float(v) for s, v in top3.items()},
    "fixed_reference_squares": [4159, 4556],
}
with open(RESULTS_DIR / "top_squares.json", "w") as f:
    json.dump(top_squares_info, f, indent=2)
top_squares_info

Top 3 squares by total internet traffic (Nov 1 - Jan 1):
square_id
5161    12740060.0
5059    11170854.0
5259    10485779.0
Name: internet_traffic, dtype: float32


{'top3_square_ids': [5161, 5059, 5259],
 'top3_totals': {5161: 12740060.0, 5059: 11170854.0, 5259: 10485779.0},
 'fixed_reference_squares': [4159, 4556]}

**Interpretation.** These three squares - the highest-traffic areas over the
full two-month window - are the ones used for every later analysis and
forecasting experiment (`01_eda.ipynb` onward), per the assignment's
instruction to identify and focus on the top-3-traffic areas. The result is
saved to `results/top_squares.json` so downstream notebooks read it back
instead of recomputing it, keeping "which squares are we forecasting" defined
in exactly one place.